### Importar Librerías

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
sns.set(color_codes=True)

_Estas librerías permiten manejar, analizar y visualizar datos de forma eficiente._

### Cargar Datos (Dataset)

In [2]:
df = pd.read_csv("C:/Users/franc/OneDrive/Desktop/TourXtreme/weather-aus/data/01_raw/weatherAUS.csv")
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RISK_MM,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,0.0,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,0.0,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,0.0,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,1.0,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,0.2,No


_Se visualizan las primeras y últimas filas del dataset para conocer su estructura. Sirve para tener una vista preliminar de la estructura de los datos, el tipo de variables, valores faltantes y posibles problemas desde el inicio._

### Visualización de los TIpos de Datos

In [3]:
df.dtypes


Date              object
Location          object
MinTemp          float64
MaxTemp          float64
Rainfall         float64
Evaporation      float64
Sunshine         float64
WindGustDir       object
WindGustSpeed    float64
WindDir9am        object
WindDir3pm        object
WindSpeed9am     float64
WindSpeed3pm     float64
Humidity9am      float64
Humidity3pm      float64
Pressure9am      float64
Pressure3pm      float64
Cloud9am         float64
Cloud3pm         float64
Temp9am          float64
Temp3pm          float64
RainToday         object
RISK_MM          float64
RainTomorrow      object
dtype: object

_Con esto podemos distinguir entre variables numéricas, categóricas y de fecha._

Este comando nos permite inspeccionar los tipos de datos de cada columna del dataset. Esto es crucial para decidir qué variables requieren transformaciones o limpiezas antes del modelado.

A continuación, se detallan los tipos detectados y las acciones recomendadas:

| Variable         | Tipo         | Requiere acción | Acción recomendada y motivo |
|------------------|--------------|------------------|-----------------------------|
| `Date`           | `object`     | ✅ Sí             | Convertir a formato fecha (`datetime`) para permitir análisis temporal. |
| `Location`       | `object`     | ⚠️ Posible        | Variable categórica. Puede requerir codificación (`LabelEncoder` o `OneHotEncoder`) si se usará en modelos. |
| `MinTemp`        | `float64`    | ❌ No             | Correcto como variable numérica. |
| `MaxTemp`        | `float64`    | ❌ No             | Correcto como variable numérica. |
| `Rainfall`       | `float64`    | ❌ No             | Correcto, requiere revisión de outliers. |
| `Evaporation`    | `float64`    | ✅ Eliminada      | Eliminada previamente por alto porcentaje de valores nulos (>40%). |
| `Sunshine`       | `float64`    | ✅ Eliminada      | Igual que `Evaporation`. Alta cantidad de valores faltantes. |
| `WindGustDir`    | `object`     | ✅ Sí             | Dirección cardinal. Puede codificarse como variable categórica. |
| `WindGustSpeed`  | `float64`    | ❌ No             | Variable numérica correcta. |
| `WindDir9am`     | `object`     | ✅ Sí             | Dirección cardinal. Requiere codificación. |
| `WindDir3pm`     | `object`     | ✅ Sí             | Dirección cardinal. Requiere codificación. |
| `WindSpeed9am`   | `float64`    | ❌ No             | Correcta. |
| `WindSpeed3pm`   | `float64`    | ❌ No             | Correcta. |
| `Humidity9am`    | `float64`    | ❌ No             | Correcta. |
| `Humidity3pm`    | `float64`    | ❌ No             | Correcta. |
| `Pressure9am`    | `float64`    | ❌ No             | Correcta. |
| `Pressure3pm`    | `float64`    | ❌ No             | Correcta. |
| `Cloud9am`       | `float64`    | ⚠️ Revisar nulos  | Revisar y tratar valores nulos si los hay. |
| `Cloud3pm`       | `float64`    | ⚠️ Revisar nulos  | Igual que `Cloud9am`. |
| `Temp9am`        | `float64`    | ❌ No             | Correcta. |
| `Temp3pm`        | `float64`    | ❌ No             | Correcta. |
| `RainToday`      | `object`     | ✅ Sí             | Ya convertida a variable binaria (`Yes`=1, `No`=0). |
| `RISK_MM`        | `float64`    | ❌ No             | Correcta. Utilizable en correlación. |
| `RainTomorrow`   | `object`     | ✅ Sí             | Ya convertida a binaria para modelado predictivo. |

_📌 Con este análisis se define qué columnas están listas para el análisis y cuáles requieren tratamiento previo, asegurando calidad y consistencia en la fase de modelado._

### Conteo de valores nulos por columna

In [4]:
df.isnull().sum()


Date                 0
Location             0
MinTemp            637
MaxTemp            322
Rainfall          1406
Evaporation      60843
Sunshine         67816
WindGustDir       9330
WindGustSpeed     9270
WindDir9am       10013
WindDir3pm        3778
WindSpeed9am      1348
WindSpeed3pm      2630
Humidity9am       1774
Humidity3pm       3610
Pressure9am      14014
Pressure3pm      13981
Cloud9am         53657
Cloud3pm         57094
Temp9am            904
Temp3pm           2726
RainToday         1406
RISK_MM              0
RainTomorrow         0
dtype: int64

_Se identifican columnas con datos faltantes._

### Porcentaje de valores nulos por columna

In [5]:
(df.isnull().mean() * 100).round(2).sort_values(ascending=False)


Sunshine         47.69
Evaporation      42.79
Cloud3pm         40.15
Cloud9am         37.74
Pressure9am       9.86
Pressure3pm       9.83
WindDir9am        7.04
WindGustDir       6.56
WindGustSpeed     6.52
WindDir3pm        2.66
Humidity3pm       2.54
Temp3pm           1.92
WindSpeed3pm      1.85
Humidity9am       1.25
Rainfall          0.99
RainToday         0.99
WindSpeed9am      0.95
Temp9am           0.64
MinTemp           0.45
MaxTemp           0.23
Location          0.00
Date              0.00
RISK_MM           0.00
RainTomorrow      0.00
dtype: float64

## Interpretación de resultados: (verrrrrr ojo 1.6 para limpieza de datos ver el pdf)

| Columna         | % de valores nulos | Acción recomendada |
|------------------|---------------------|---------------------|
| `Sunshine`       | 47.69%              | ❌ Eliminar. Alta cantidad de nulos, difícil imputación precisa. |
| `Evaporation`    | 42.79%              | ❌ Eliminar. Igual que anterior. |
| `Cloud3pm`       | 40.15%              | ❌ Eliminar. Incompleto y poco fiable. |
| `Cloud9am`       | 37.74%              | ❌ Eliminar. Misma razón. |
| `Pressure9am`    | 9.86%               | ⚠️ Imputar media o interpolar. Se puede recuperar. |
| `Pressure3pm`    | 9.83%               | ⚠️ Igual que anterior. |
| `WindDir9am`     | 7.04%               | ⚠️ Imputar modo o codificar con “Desconocido”. |
| `WindGustDir`    | 6.56%               | ⚠️ Igual. No eliminar. |
| `WindGustSpeed`  | 6.52%               | ⚠️ Imputar con media o interpolar. Variable importante. |
| `WindDir3pm`     | 2.66%               | ✅ Imputar. Valor categórico. |
| `Humidity3pm`    | 2.54%               | ✅ Imputar. Es bajo. |
| `Temp3pm`        | 1.92%               | ✅ Imputar. Importante para modelar temperatura futura. |
| `WindSpeed3pm`   | 1.85%               | ✅ Imputar. Bajo impacto. |
| `Humidity9am`    | 1.25%               | ✅ Imputar. |
| `Rainfall`       | 0.99%               | ✅ Imputar o eliminar filas puntuales. |
| `RainToday`      | 0.99%               | ✅ Convertida a binaria. Imputar como 0 si no llovió. |
| `WindSpeed9am`   | 0.95%               | ✅ Imputar. |
| `Temp9am`        | 0.64%               | ✅ Imputar. |
| `MinTemp`        | 0.45%               | ✅ Imputar. |
| `MaxTemp`        | 0.23%               | ✅ Imputar. |
| `Location`       | 0.00%               | Sin nulos. OK. |
| `Date`           | 0.00%               | Sin nulos. OK. |
| `RISK_MM`        | 0.00%               | Sin nulos. OK. |
| `RainTomorrow`   | 0.00%               | Sin nulos. OK. Convertida a binaria. |

Se definió un umbral del 40% de datos faltantes para eliminar columnas debido a su bajo valor analítico. Superado este límite, la imputación deja de ser confiable, ya que no existen suficientes observaciones para reconstruir patrones significativos. Mantener estas variables puede introducir sesgos o ruido en los modelos predictivos.

Por lo tanto, se eliminaron las columnas Sunshine, Evaporation, Cloud3pm y Cloud9am por superar este porcentaje.

## NOTA

Media o mediana: para columnas numéricas.
Ejemplo: si Pressure9am tiene valores nulos, se puede usar el promedio de esa columna para completar los vacíos.

Moda o categoría “Desconocido”: para variables categóricas.
Ejemplo: si WindGustDir tiene valores nulos, se puede imputar con la dirección más frecuente o con un valor genérico como “Unknown”.

Interpolación: si los datos son secuenciales (como una serie temporal), se pueden rellenar con el promedio entre valores vecinos.

¿Por qué es importante?
Imputar ayuda a conservar más datos, lo que mejora la precisión del modelo. Sin embargo, si una columna tiene demasiados nulos (como >40%), la imputación podría distorsionar los resultados, por eso se eliminan esas columnas.